# 01 — Integrate PriceCatcher data

**Start here:** select the project's Python kernel, then **Run All**. Each manual execution of 01 starts a separate run and remembers it for Notebooks 02–06. Continue in order with Run All; no path or mode changes are needed. Running 01 again starts another reproduction, preserving earlier experiments.

Combine January 2025–June 2026, attach item/premise details and standardise egg units. A verified cached source is copied when available, so monthly downloads are unnecessary. Without a cache, official-source access is required. Failed downloads never export a partial combined dataset.

The same source and lookup snapshots are shared by every fold. Lookup snapshots do not establish historical shop metadata. Generated files stay in `data/`; the current run folder is printed below. Saved outputs describe the last local execution.

Install dependencies once using `requirements.txt`. Keep `pricecatcher_config.py` and `prepare.py` beside the six notebooks. The terminal runner remains optional.

Repeating the exposed April–June 2026 test is a reproduction, not new independent validation.

In [ ]:
import pandas as pd
import json
import hashlib
from datetime import datetime, timezone
import pricecatcher_config as cfg
from importlib import reload
cfg = reload(cfg)
cfg.start_notebook_run()
cfg = reload(cfg)
cfg.protect_frozen_preparation()
print("Run folder:", cfg.PROJECT_ROOT)
print("Experiment:", cfg.EXPERIMENT, "| Training cutoff:", cfg.TRAIN_END_DATE)

reuse_source = (cfg.STAGE1_COMBINED_PARQUET.exists()
                and (cfg.STAGE1_DIR / "source_manifest.json").exists())
if reuse_source:
    cached_manifest = cfg.verify_source_snapshot()
    expected_months = pd.period_range(cfg.START_MONTH, cfg.END_MONTH, freq="M").astype(str).tolist()
    if cached_manifest["months"] != expected_months or cached_manifest["selected_item_codes"] != cfg.SCOPE_ITEM_CODES:
        raise ValueError("Cached source scope differs. Use a new empty data folder for the changed scope.")


## 1. Load lookup snapshots
 Local snapshots are reused so later runs use the same item and premise descriptions. Joins must have unique codes. The egg lookup units must agree with the conversion rule.


In [6]:
item_path = cfg.LOOKUP_DIR / "lookup_item.parquet"
premise_path = cfg.LOOKUP_DIR / "lookup_premise.parquet"

# Reuse these snapshots on later runs so joins remain reproducible.
if not item_path.exists():
    pd.read_parquet(cfg.URL_ITEM).to_parquet(item_path, index=False)
if not premise_path.exists():
    pd.read_parquet(cfg.URL_PREMISE).to_parquet(premise_path, index=False)

lookup_saved_utc = datetime.fromtimestamp(item_path.stat().st_mtime, timezone.utc).isoformat()
item = pd.read_parquet(item_path)
premise = pd.read_parquet(premise_path)
item["item_code"] = pd.to_numeric(item["item_code"], errors="raise").astype("Int64")
premise["premise_code"] = pd.to_numeric(premise["premise_code"], errors="raise").astype("Int64")

for table, key in [(item, "item_code"), (premise, "premise_code")]:
    if table[key].isna().any() or table[key].duplicated().any():
        raise ValueError(f"Lookup must have unique, non-missing {key} values.")

# Confirm the fixed egg conversion still agrees with this lookup snapshot.
egg_units = item.set_index("item_code")["unit"].astype("string").str.strip().str.lower()
for code in cfg.ALL_EGG_CODES:
    expected = "30 biji" if code in cfg.EGG_30_CODES else "10 biji"
    if code not in egg_units.index or egg_units.loc[code] != expected:
        raise ValueError(f"Check egg unit for item {code}: expected {expected}.")

item = item[["item_code", "item", "unit", "item_group", "item_category"]]
premise = premise[["premise_code", "premise", "state", "district", "premise_type"]]
print("Lookup checks passed.")


Lookup checks passed.


## 2. Define the monthly integration step
 Keep the selected basket, check dates and joins, then convert 30-egg packs to a 10-egg equivalent. Prices remain in the column `price`.


In [7]:
def integrate_month(month):
    url = cfg.BASE_PRICE_URL.format(month=month)
    columns = ["date", "premise_code", "item_code", "price"]
    prices = pd.read_parquet(url, columns=columns)

    prices["item_code"] = pd.to_numeric(prices["item_code"], errors="raise").astype("Int64")
    prices = prices[prices["item_code"].isin(cfg.SCOPE_ITEM_CODES)].copy()
    prices["premise_code"] = pd.to_numeric(prices["premise_code"], errors="raise").astype("Int64")
    prices["date"] = pd.to_datetime(prices["date"], errors="raise").dt.normalize()
    prices["price"] = pd.to_numeric(prices["price"], errors="coerce")
    if prices.empty or prices["date"].isna().any():
        raise ValueError(f"{month}: no selected records or missing dates.")
    if not prices["date"].dt.to_period("M").eq(pd.Period(month)).all():
        raise ValueError(f"{month}: records outside the expected month.")

    prices["project_group"] = prices["item_code"].map(cfg.PROJECT_ITEM_MAP)
    data = prices.merge(item, on="item_code", how="left", validate="many_to_one")
    data = data.merge(premise, on="premise_code", how="left", validate="many_to_one")
    for col in cfg.TEXT_COLS:
        data[col] = data[col].astype("string").str.strip().replace("", pd.NA)
    if data[["premise_code", "item_code"] + cfg.TEXT_COLS].isna().any().any():
        raise ValueError(f"{month}: missing codes or lookup details; inspect before continuing.")

    data.loc[data["item_code"].isin(cfg.EGG_30_CODES), "price"] /= 3
    data.loc[data["item_code"].isin(cfg.ALL_EGG_CODES), "unit"] = "10 biji"
    data = data[cfg.FINAL_COLS].copy()
    if data.duplicated(["date", "premise_code", "item_code"]).any():
        raise ValueError(f"{month}: duplicate date-premise-item keys need review.")
    return data


## 3. Combine all months and save
 No monthly CSVs or duplicate full CSV are saved. Lookups downloaded today are a snapshot, not historical premise metadata; acknowledge that limit in the report.


In [8]:
if reuse_source:
    print("Reusing verified integrated source:", cfg.STAGE1_COMBINED_PARQUET)
    print("Combined rows:", cached_manifest["rows"])
    print("No monthly source download or source rewrite needed.")
    print("Next: Notebook 02.")
else:
    months = pd.period_range(cfg.START_MONTH, cfg.END_MONTH, freq="M").astype(str).tolist()
    frames = []
    summary = []
    for month in months:
        # Errors stop execution; do not export a partial combined dataset.
        data = integrate_month(month)
        frames.append(data)
        summary.append({"month": month, "rows": len(data),
                        "date_min": data["date"].min(), "date_max": data["date"].max()})
        print(month, "rows:", len(data))
    
    combined = pd.concat(frames, ignore_index=True)
    del frames
    combined.to_parquet(cfg.STAGE1_COMBINED_PARQUET, index=False)
    source_files = {}
    for path in [cfg.STAGE1_COMBINED_PARQUET, item_path, premise_path]:
        with path.open("rb") as handle:
            source_files[str(path.relative_to(cfg.STAGE1_DIR))] = hashlib.file_digest(handle, "sha256").hexdigest()
    source_snapshot = hashlib.sha256(json.dumps(source_files, sort_keys=True).encode()).hexdigest()
    manifest = {
        "source_files_sha256": source_files, "source_snapshot_sha256": source_snapshot,
        "run_time_utc": datetime.now(timezone.utc).isoformat(),
        "lookup_snapshot_saved_utc": lookup_saved_utc,
        "item_source": cfg.URL_ITEM, "premise_source": cfg.URL_PREMISE,
        "price_source": cfg.BASE_PRICE_URL,
        "months": months, "selected_item_codes": cfg.SCOPE_ITEM_CODES,
        "source_start_month": cfg.START_MONTH, "source_end_month": cfg.END_MONTH,
        "monthly_counts": [{"month": row["month"], "rows": row["rows"]} for row in summary],
        "rows": len(combined), "pandas_version": pd.__version__,
        "date_min": str(combined["date"].min().date()),
        "date_max": str(combined["date"].max().date()),
        "note": "Lookup snapshots are reused; monthly source files may be revised by the publisher.",
        "egg_conversion": "30 biji prices divided by 3; egg unit stored as 10 biji",
    }
    (cfg.STAGE1_DIR / "source_manifest.json").write_text(json.dumps(manifest, indent=2))
    print("Combined rows:", len(combined))
    print("Saved:", cfg.STAGE1_COMBINED_PARQUET)
    print("Next: Notebook 02.")


Reusing verified integrated source: /Users/alfredyap/Documents/GitHub/FYP/data/01_integrated/integrated_prices.parquet
Combined rows: 4731934
No monthly source download or source rewrite needed.
Next: Notebook 02.
